# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hassan-tech-pro/ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

These are constructive questions about what the disclosed design can support, not challenges to the authors' good-faith analysis. I’m using the March 2026 paper excerpt supplied with this task; I have not re-created its portfolio calculations.

**Finding 1 — “The Anatomy of Growing Content” (reported as confirmed).** The paper reports 74,187 rising versus 45,272 falling pages; the rising group averages 3,180 words and 184 days old, versus 2,311 words and 230 days for the falling group. The outcome direction is based on a 30-day impression change, so my first question is: what exact thresholds and eligible-page rules define “rising” and “falling,” and are repeated pages/brands handled as clustered observations? A second question is whether client mix, topic mix, or publication cohorts could explain part of the age/length difference. These are useful descriptive contrasts, but by themselves they do not show that adding words or changing age causes growth.

**Finding 2 — “The Freshness Multiplier” (reported as confirmed).** The paper reports a 7.88:1 growth-to-decline ratio in the 31–90-day freshness band, and a 3.2× health / 57× impressions comparison for 365+ day pages refreshed within 30 days. The paper also notes the unstable 361+ ratio (283 growing pages versus one declining page) and describes the study as observational. My methodology question is: were refreshed and comparison pages aligned on pre-refresh age, prior visibility, content quality, client, and topic, with a clearly later outcome window? Without that design, the measured differences are informative associations, but the word “effect” should not be read as a causal estimate.

*For each finding, ask where the outcome comes from and whether the disclosed comparison design carries the wording.*

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_RELATIVE_PATH = Path("content_refresh_anonymized.csv")
repo_candidates = [Path.cwd(), *Path.cwd().parents]
data_path = next(
    (root / DATA_RELATIVE_PATH for root in repo_candidates if (root / DATA_RELATIVE_PATH).is_file()),
    None,
)
DATA_URL = (
    "https://raw.githubusercontent.com/Hassan-tech-pro/ml-internship/main/"
    "data/raw/content_refresh_anonymized.csv"
)
if data_path is not None:
    raw_data = pd.read_csv(data_path)
    repo_root = data_path.resolve().parents[2]
else:
    raw_data = pd.read_csv(DATA_URL)
    repo_root = Path.cwd()

required_columns = {
    "content_id",
    "client_id",
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "trend_direction",
    "trend_pct",
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "content_type",
    "main_intent",
}
missing_columns = required_columns.difference(raw_data.columns)
if missing_columns:
    raise ValueError(f"Starter CSV is missing required columns: {sorted(missing_columns)}")

numeric_columns = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "trend_pct",
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
]
for column in numeric_columns:
    raw_data[column] = pd.to_numeric(raw_data[column], errors="coerce")

pages = (
    raw_data.loc[
        raw_data["impressions_90d"].gt(0)
        & raw_data["content_age_days"].ge(90)
    ]
    .drop_duplicates(subset="content_id")
    .copy()
)
pages["decline_proxy"] = (
    pages["trend_direction"].fillna("").astype(str).str.lower().eq("down").astype("int8")
)

numeric_features = [
    "content_age_days",
    "days_since_last_update",
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
]
categorical_features = ["content_type", "main_intent"]
feature_columns = numeric_features + categorical_features
target_column = "decline_proxy"
group_column = "client_id"

forbidden_features = {
    "trend_direction",
    "trend_pct",
    target_column,
    "content_id",
    "client_id",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "impressions_last_30d",
    "impressions_prev_30d",
}
assert forbidden_features.isdisjoint(feature_columns)

print(f"Source: {'local starter CSV' if data_path is not None else 'public starter CSV'}")
print(f"Eligible unique pages: {len(pages):,}; client groups: {pages[group_column].nunique():,}")
print(f"Current decline-proxy prevalence: {pages[target_column].mean():.1%}")
print(f"Model features: {', '.join(feature_columns)}")
print("Proxy is the current trend bucket, not an independently measured future outcome.")

Source: public starter CSV
Eligible unique pages: 30,000; client groups: 32
Current decline-proxy prevalence: 54.2%
Model features: content_age_days, days_since_last_update, search_volume, competition, cpc, word_count, char_count, content_type, main_intent
Proxy is the current trend bucket, not an independently measured future outcome.


## 2. My model under an honest split (before/after)

**Before:** reproduce the Week-5 Logistic Regression under a stratified random 75/25 split, which can place pages from the same client on both sides and is optimistic for transfer to unseen clients.

**After:** use a 75/25 `GroupShuffleSplit` by `client_id`, selecting a deterministic seed that leaves both proxy classes in train and test. This is the primary honest estimate for a new-client use case. The split is not time-aware, and the starter target remains contemporaneous, so neither estimate predicts future decline.

For each split, score both Logistic Regression and the Week-4 stale-visible rule on the same held-out rows, using Precision@10, Precision@50, average precision, and the held-out proxy rate. Compare random versus grouped results as a sensitivity check, not proof of a specific source of bias.

In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, average_precision_score, balanced_accuracy_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

all_indices = np.arange(len(pages))
random_train, random_test = train_test_split(
    all_indices,
    test_size=0.25,
    random_state=42,
    stratify=pages[target_column],
)

group_split = None
for group_seed in (42, 17, 11, 7, 23):
    candidate_split = next(
        GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=group_seed).split(
            pages[feature_columns],
            pages[target_column],
            groups=pages[group_column],
        )
    )
    candidate_train, candidate_test = candidate_split
    if (
        pages.iloc[candidate_train][target_column].nunique() == 2
        and pages.iloc[candidate_test][target_column].nunique() == 2
    ):
        group_split = candidate_split
        break
if group_split is None:
    raise ValueError("Could not create a client-held-out split with both proxy classes.")

group_train, group_test = group_split
assert set(pages.iloc[group_train][group_column]).isdisjoint(
    set(pages.iloc[group_test][group_column])
)


def build_model():
    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
            ("scaler", StandardScaler()),
        ]
    )
    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]
    )
    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", numeric_pipeline, numeric_features),
            ("categorical", categorical_pipeline, categorical_features),
        ]
    )
    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("classifier", LogisticRegression(C=1.0, max_iter=2000, random_state=42)),
        ]
    )


def precision_at_k(y_true, scores, k):
    truth = np.asarray(y_true, dtype=int)
    ranking_scores = np.asarray(scores, dtype=float)
    if len(truth) < k:
        raise ValueError(f"Precision@{k} requires at least {k} test rows; found {len(truth)}.")
    order = np.argsort(-ranking_scores, kind="mergesort")[:k]
    return float(truth[order].mean())


def week4_rule_scores(test_pages):
    eligible_for_rule = (
        test_pages["impressions_90d"].ge(300)
        & test_pages["days_since_last_update"].ge(180)
    )
    return np.where(
        eligible_for_rule,
        test_pages["impressions_90d"].fillna(0).to_numpy(dtype=float),
        0.0,
    )


def evaluate_split(split_name, train_indices, test_indices):
    train_pages = pages.iloc[train_indices]
    test_pages = pages.iloc[test_indices]
    X_train = train_pages[feature_columns]
    y_train = train_pages[target_column].astype(int)
    X_test = test_pages[feature_columns]
    y_test = test_pages[target_column].astype(int)

    model = build_model()
    model.fit(X_train, y_train)
    model_scores = model.predict_proba(X_test)[:, list(model.classes_).index(1)]
    rule_scores = week4_rule_scores(test_pages)

    rows = []
    for method_name, scores in [
        ("Week-4 stale-visible rule", rule_scores),
        ("Week-5 Logistic Regression", model_scores),
    ]:
        rows.append(
            {
                "split": split_name,
                "method": method_name,
                "test_pages": len(test_pages),
                "test_clients": test_pages[group_column].nunique(),
                "proxy_rate": float(y_test.mean()),
                "Precision@10": precision_at_k(y_test, scores, 10),
                "Precision@50": precision_at_k(y_test, scores, 50),
                "Average precision": average_precision_score(y_test, scores),
                "Accuracy": accuracy_score(y_test, scores >= 0.5),
                "Balanced accuracy": balanced_accuracy_score(y_test, scores >= 0.5),
            }
        )
    return rows, model, test_pages, y_test, model_scores


random_results, random_model, random_test_pages, random_y_test, random_model_scores = evaluate_split(
    "Before: random page split", random_train, random_test
)
grouped_results, grouped_model, grouped_test_pages, grouped_y_test, grouped_model_scores = evaluate_split(
    "After: client-held-out split", group_train, group_test
)
validation_table = pd.DataFrame(random_results + grouped_results)
display(validation_table)

random_model_ap = validation_table.loc[
    (validation_table["split"] == "Before: random page split")
    & (validation_table["method"] == "Week-5 Logistic Regression"),
    "Average precision",
].iloc[0]
grouped_model_ap = validation_table.loc[
    (validation_table["split"] == "After: client-held-out split")
    & (validation_table["method"] == "Week-5 Logistic Regression"),
    "Average precision",
].iloc[0]
print(
    "Random-minus-grouped average-precision gap (model): "
    f"{random_model_ap - grouped_model_ap:+.3f}. Treat this as split sensitivity, not a causal diagnosis."
)
print(f"Client-grouped split seed: {group_seed}; train/test clients do not overlap.")

,split,method,test_pages,test_clients,proxy_rate,Precision@10,Precision@50,Average precision,Accuracy,Balanced accuracy
0,Before: random page split,Week-4 stale-visible rule,7500,31,0.542000,0.6,0.58,0.542225,0.458267,0.500246
1,Before: random page split,Week-5 Logistic Regression,7500,31,0.542000,0.5,0.54,0.639705,0.607467,0.591006
2,After: client-held-out split,Week-4 stale-visible rule,7115,8,0.516514,0.8,0.70,0.517041,0.483907,0.500399
3,After: client-held-out split,Week-5 Logistic Regression,7115,8,0.516514,0.8,0.64,0.528778,0.530710,0.524755


Random-minus-grouped average-precision gap (model): +0.111. Treat this as split sensitivity, not a causal diagnosis.
Client-grouped split seed: 42; train/test clients do not overlap.


## 3. Leakage audit

The starter target is defined from `trend_direction`, and `trend_direction` is derived from `trend_pct`; neither may be a model feature. IDs are context for grouping only. The remaining metadata fields do not directly encode the target definition, but their exact measurement timestamps are not available in this cross-sectional file. In particular, freshness is measured at the snapshot date, so a future-outcome model would need a clearly frozen feature cutoff.

The code below checks the final Week-5 feature list and deliberately trains a one-feature `trend_pct` model on the grouped split as a leakage challenge. That score is invalid and must never be used as a result; it demonstrates how access to the label's source can inflate apparent performance.

In [3]:
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier

label_source_columns = {"trend_direction", "trend_pct", target_column}
identifier_columns = {"content_id", "client_id"}
performance_window_columns = {
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "impressions_last_30d",
    "clicks_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
}

assert label_source_columns.isdisjoint(feature_columns)
assert identifier_columns.isdisjoint(feature_columns)
assert performance_window_columns.isdisjoint(feature_columns)

feature_audit = pd.DataFrame(
    [
        {
            "field": "trend_direction / trend_pct",
            "role": "label source / direct leakage risk",
            "used_by_clean_model": False,
            "reason": "The starter decline proxy is defined from this trend bucket and percentage.",
        },
        {
            "field": "content_id / client_id",
            "role": "context only",
            "used_by_clean_model": False,
            "reason": "Pseudonymous identifiers are used only for deduplication and client grouping.",
        },
        {
            "field": "impressions / clicks / sessions",
            "role": "overlapping performance-window risk",
            "used_by_clean_model": False,
            "reason": "These current snapshot performance totals are excluded from the Week-5 feature set.",
        },
        {
            "field": "age, freshness, keyword/content metadata",
            "role": "candidate feature set",
            "used_by_clean_model": True,
            "reason": "No direct label-source field; exact within-window timestamps remain unavailable.",
        },
    ]
)
display(feature_audit)

clean_grouped_ap = average_precision_score(grouped_y_test, grouped_model_scores)
leak_model = make_pipeline(
    SimpleImputer(strategy="median"),
    DecisionTreeClassifier(max_depth=1, random_state=42),
)
leak_train_x = pages.iloc[group_train][["trend_pct"]]
leak_test_x = pages.iloc[group_test][["trend_pct"]]
leak_train_y = pages.iloc[group_train][target_column].astype(int)
leak_model.fit(leak_train_x, leak_train_y)
leak_scores = leak_model.predict_proba(leak_test_x)[:, list(leak_model.classes_).index(1)]
leaky_grouped_ap = average_precision_score(grouped_y_test, leak_scores)
leak_challenge = pd.DataFrame(
    [
        {"model": "Clean Week-5 features", "grouped_average_precision": clean_grouped_ap},
        {"model": "INVALID: trend_pct leakage challenge", "grouped_average_precision": leaky_grouped_ap},
    ]
)
display(leak_challenge)
print(
    "The trend_pct model is deliberately invalid because trend_pct defines the target proxy. "
    "It is shown only as a leakage diagnostic and is not part of the model comparison or claim."
)
del leak_model, leak_train_x, leak_test_x, leak_train_y, leak_scores, leak_challenge

,field,role,used_by_clean_model,reason
0,trend_direction / trend_pct,label source / direct leakage risk,False,The starter decline proxy is defined from this...
1,content_id / client_id,context only,False,Pseudonymous identifiers are used only for ded...
2,impressions / clicks / sessions,overlapping performance-window risk,False,These current snapshot performance totals are ...
3,"age, freshness, keyword/content metadata",candidate feature set,True,No direct label-source field; exact within-win...


,model,grouped_average_precision
0,Clean Week-5 features,0.528778
1,INVALID: trend_pct leakage challenge,0.893436


The trend_pct model is deliberately invalid because trend_pct defines the target proxy. It is shown only as a leakage diagnostic and is not part of the model comparison or claim.


## 4. Claim rewrite and error review

**Overclaim to avoid:** “The model predicts which pages will decline and tells editors what to refresh.”

**Evidence-aligned rewrite:** “On this starter snapshot, the metadata-only Logistic Regression ranks pages against a contemporaneous trend-derived decline proxy. Its client-held-out metrics measure agreement with that proxy, not future decline or refresh impact. At most, the score is a directional decision-support cue for human review; a future-window label and later time-based validation are needed before operational use.”

The code reports anonymous false-positive/false-negative examples and grouped permutation importance from the client-held-out test set. The example fields are measurements, not page identifiers; importance is descriptive, not causal.

In [4]:
grouped_predictions = (grouped_model_scores >= 0.5).astype(int)
grouped_error_review = grouped_test_pages[
    [
        "content_type",
        "content_age_days",
        "days_since_last_update",
        "search_volume",
        "word_count",
    ]
].reset_index(drop=True)
grouped_error_review["proxy_true"] = grouped_y_test.reset_index(drop=True)
grouped_error_review["model_probability"] = grouped_model_scores
grouped_error_review["model_prediction_at_0_5"] = grouped_predictions
grouped_error_review["error_type"] = np.select(
    [
        (grouped_error_review["model_prediction_at_0_5"] == 1)
        & (grouped_error_review["proxy_true"] == 0),
        (grouped_error_review["model_prediction_at_0_5"] == 0)
        & (grouped_error_review["proxy_true"] == 1),
    ],
    ["False positive", "False negative"],
    default="Correct",
)

incorrect_rows = grouped_error_review.loc[
    grouped_error_review["error_type"] != "Correct"
].copy()
if incorrect_rows.empty:
    print("No threshold errors at 0.5 in this grouped holdout; no failure examples to display.")
else:
    failure_examples = (
        incorrect_rows.assign(
            confidence_distance=(incorrect_rows["model_probability"] - 0.5).abs()
        )
        .sort_values("confidence_distance", ascending=False)
        .head(3)
        .drop(columns="confidence_distance")
        .copy()
    )
    failure_examples["why_hard"] = np.where(
        failure_examples["error_type"].eq("False positive"),
        "Metadata looked decline-associated, but the trend-derived proxy was not down.",
        "The metadata pattern did not separate this down-proxy page from other pages.",
    )
    print(f"Anonymous high-confidence errors (up to 3 of {len(incorrect_rows):,}):")
    display(
        failure_examples[
            [
                "error_type",
                "content_type",
                "content_age_days",
                "days_since_last_update",
                "search_volume",
                "word_count",
                "model_probability",
                "why_hard",
            ]
        ]
    )

error_rates_by_type = (
    grouped_error_review.groupby("content_type", dropna=False)
    .agg(
        n=("proxy_true", "size"),
        proxy_rate=("proxy_true", "mean"),
        threshold_error_rate=("error_type", lambda values: values.ne("Correct").mean()),
    )
    .reset_index()
    .sort_values("n", ascending=False)
)
print("Grouped-test error rates by content type; interpret small groups cautiously.")
display(error_rates_by_type)

importance = permutation_importance(
    grouped_model,
    pages.iloc[group_test][feature_columns],
    grouped_y_test,
    scoring="average_precision",
    n_repeats=5,
    random_state=42,
    n_jobs=-1,
)
grouped_importance = (
    pd.DataFrame(
        {
            "feature": feature_columns,
            "mean_average_precision_drop": importance.importances_mean,
            "std_average_precision_drop": importance.importances_std,
        }
    )
    .sort_values("mean_average_precision_drop", ascending=False)
    .reset_index(drop=True)
)
print("Top held-out permutation-importance signals; these are not causal effects.")
display(grouped_importance.head(5))
print("Claim rewrite: use the safe wording in Section 4; do not call this future-decline prediction.")

Anonymous high-confidence errors (up to 3 of 3,339):


,error_type,content_type,content_age_days,days_since_last_update,search_volume,word_count,model_probability,why_hard
3336,False negative,keyword article,502,20,NaN,NaN,0.121672,The metadata pattern did not separate this dow...
3302,False negative,keyword article,502,20,NaN,NaN,0.121672,The metadata pattern did not separate this dow...
5739,False negative,keyword article,502,22,NaN,NaN,0.122795,The metadata pattern did not separate this dow...


Grouped-test error rates by content type; interpret small groups cautiously.


,content_type,n,proxy_rate,threshold_error_rate
1,keyword article,6418,0.510439,0.473824
0,comparison article,697,0.572453,0.427547


Top held-out permutation-importance signals; these are not causal effects.


,feature,mean_average_precision_drop,std_average_precision_drop
0,content_age_days,0.022798,0.003480
1,word_count,0.003592,0.001786
2,competition,0.000630,0.001168
3,search_volume,0.000318,0.001007
4,cpc,0.000147,0.001016


Claim rewrite: use the safe wording in Section 4; do not call this future-decline prediction.


## Self-check

Leave execution-dependent items unchecked until you run the notebook top to bottom and review its outputs:

- [x] Two paper findings are summarized with constructive methodology questions using the supplied PDF
- [x] Week-5 features, target, eligibility filters, and baseline rule are reproduced in code
- [x] The random-versus-client-held-out comparison uses the same proxy, metrics, and Week-4 rule
- [x] Direct label-source fields, IDs, and performance-window totals are excluded from clean features
- [x] A deliberately invalid label-derived feature is identified and excluded from the honest result
- [x] Claim wording is limited to observed proxy agreement and decision-support language
- [ ] Notebook runs top to bottom; split metrics, leakage challenge, errors, and importance are inspected
- [ ] No row identifiers or sensitive values appear in saved outputs
- [ ] Save locally under `work/notebooks/`; publish only when you choose